# Clase 2. Diseño de Herramientas e Integración de MCP

## Tools predefinidas

| Herramienta | Qué hace |
| :--- | :--- |
| **Read** | Leer cualquier archivo en el directorio de trabajo |
| **Write** | Crear nuevos archivos |
| **Edit** | Realizar ediciones precisas en archivos existentes |
| **Bash** | Ejecutar comandos de terminal, scripts, operaciones de git |
| **Monitor** | Observar un script de fondo y reaccionar a cada línea de salida como un evento |
| **Glob** | Encontrar archivos por patrón (`**/*.ts`, `src/**/*.py`) |
| **Grep** | Buscar contenido de archivos con expresiones regulares |
| **WebSearch** | Buscar en la web información actual |
| **WebFetch** | Obtener y analizar contenido de páginas web |
| **AskUserQuestion** | Hacer preguntas aclaratorias al usuario con opciones de opción múltiple |

---

## 2. Diseño de Tools Efectivos

### Tool Descriptions: el mecanismo de selección
Cuando Claude tiene acceso a múltiples tools, la description es el mecanismo principal que usa para decidir cuál invocar. No es el nombre, no es el schema — es la description la que determina si el modelo selecciona correctamente.

El problema de descriptions mínimas.  
```json
{
  "name": "analyze_content",
  "description": "Analyzes content"
}
```  

Con una description tan escueta, Claude no puede distinguir de forma confiable entre tools similares. Si además tienes analyze_document con description “Analyzes a document”, el modelo va a hacer misrouting constantemente.

### Qué incluir en una buena description

**Una description efectiva contiene:**

1. Qué hace el tool — propósito específico, no genérico
2. Input formats — qué tipo de datos acepta
3. Example queries — cuándo debería invocarse
4. Edge cases — qué NO hacer con este tool
5. Boundary explanations — dónde termina su responsabilidad

```json
{
  "name": "analyze_sentiment",
  "description": "Analyzes emotional sentiment of customer feedback text. Input: raw text string from support tickets or reviews (max 5000 chars). Use this for: determining if feedback is positive/negative/neutral, extracting emotion keywords. Do NOT use for: document summarization, content classification by topic, or structured data analysis. Returns: sentiment score (-1 to 1), dominant emotion, confidence level."
}
```

---

### Overlap ambiguo entre tools
#### El problema
Cuando dos tools tienen nombres o descriptions que se superponen, Claude no puede determinar cuál usar:

| Tool A | Tool B | Problema / Desafío |
| :--- | :--- | :--- |
| `analyze_content` | `analyze_document` | ¿Cuál es para texto plano vs. PDF? *(Falta de claridad en el tipo de entrada)* |
| `search_records` | `find_entries` | Sinónimos sin distinción clara *(Duplicidad de funciones)* |
| `process_data` | `transform_data` | Verbos genéricos con la misma semántica *(Ambigüedad en la acción)* |


#### Diagnóstico de misrouting
Si observas que Claude invoca el tool equivocado:
1. Revisa las descriptions — ¿hay palabras compartidas que crean ambigüedad?
2. Compara los schemas de input — ¿aceptan los mismos tipos?
3. Verifica si el system prompt crea asociaciones no intencionadas

---

### System prompt y asociaciones no deseadas
El wording del system prompt puede crear asociaciones implícitas con tools. Si tu prompt dice:

`Cuando el usuario pida analizar algo, usa las herramientas disponibles`

Claude puede asociar `analizar` con cualquier tool que tenga `analyze` en el nombre, ignorando la intención real.

**Solución:** ser específico en el system prompt sobre cuándo usar cada tool:

`Para análisis de sentimiento de feedback, usa analyze_sentiment. Para clasificación de documentos por categoría, usa classify_document.`

---

### Estrategias para eliminar overlap

1. **Renombrar tools con verbos específicos:**   
    ❌ analyze_content → ✅ extract_sentiment.   
    ❌ analyze_document → ✅ classify_document_type.   
    ❌ process_data → ✅ normalize_csv_records.   
2. **Actualizar descriptions al renombrar:** Renombrar sin actualizar la description no resuelve nada. El nombre es una señal secundaria; la description es la primaria.

3. **Splittear tools genéricos:** Un tool genérico que hace “de todo” es peor que varios tools específicos con contratos claros.

## 3. Conectar con herramientas externas usando MCP

### MCP Server Scoping

Los MCP servers se configuran en dos niveles con alcances diferentes:

**Project-level:** .mcp.json
Archivo en la raíz del proyecto, versionado con git. Todos los miembros del equipo comparten esta configuración:
```json
{
  "mcpServers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_TOKEN": "${GITHUB_TOKEN}"
      }
    }
  }
}
```
La sintaxis ${GITHUB_TOKEN} expande variables de entorno del shell del usuario. Esto permite compartir la configuración sin exponer secretos — cada desarrollador tiene su propio token en su environment.

**User-level:** ~/.claude.json
Configuración personal, no compartida via version control. Útil para tools que solo un usuario necesita:
```json
{
  "mcpServers": {
    "personal-notes": {
      "command": "node",
      "args": ["/home/user/tools/notes-server.js"]
    }
  }
}
```


In [6]:
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

options = ClaudeAgentOptions(
    model = "claude-sonnet-4-6",
    mcp_servers={
        "claude-code-docs": {
            "type": "http",
            "url": "https://code.claude.com/docs/mcp",
        }
    },
    allowed_tools=["mcp__claude-code-docs__*"],
)

In [7]:
async for message in query(
    prompt="Use the docs MCP server to explain what hooks are in Claude Code",
    options=options,
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

Here's a comprehensive explanation of **hooks in Claude Code**, drawn directly from the official docs:

---

## What Are Hooks?

Hooks are **user-defined shell commands, HTTP endpoints, LLM prompts, or agents** that execute automatically at specific points in Claude Code's lifecycle. They give you **deterministic control** over Claude's behavior — ensuring certain actions always happen rather than relying on Claude to choose to do them.

---

## Hook Types

There are 5 kinds of hook handlers:

| Type | Description |
|---|---|
| `"command"` | Run a shell script. Receives event JSON on stdin, communicates back via exit codes and stdout |
| `"http"` | POST the event JSON to an HTTP endpoint |
| `"mcp_tool"` | Call a tool on an already-connected MCP server |
| `"prompt"` | Send a prompt to a Claude model for a yes/no decision |
| `"agent"` | Spawn a subagent that can use tools (Read, Grep, etc.) before returning a decision *(experimental)* |

---

## Lifecycle Events

Hooks fire at three c

### Caso real: conflicto de modelo principal vs. advisorModel

Al intentar ejecutar un `query()` del Agent SDK con el modelo **opus** como
modelo principal, la llamada falló con este error:

> `API Error: 400 tools.10.model: 'claude-sonnet-4-6' cannot be used as an
> advisor when the request model is 'claude-opus-4-8'.`

#### Qué pasó

El Agent SDK no solo usa el modelo principal de la request: también puede
correr **subagentes / advisors** con su propio modelo. En este caso había una
configuración heredada del `settings.json` de usuario (`~/.claude/settings.json`):

​```json
"model": "opus",          // request principal  → claude-opus-4-8
"advisorModel": "sonnet"  // advisor/subagente  → claude-sonnet-4-6
​```

Al armar la petición, el SDK combinó **opus (principal)** con un **advisor en
sonnet**. La API rechaza esa combinación: un advisor no puede usar un modelo
distinto al de la request principal en ese escenario → error `400`.

Punto clave: el error **no venía del notebook ni del servidor MCP**. El
`tools.10` del mensaje es el agente/advisor inyectado por la configuración,
no una tool del MCP. Por eso el mismo error aparecía tanto en el ejemplo de
subagentes como en el de MCP: la causa estaba en la config global, no en el código.

#### Causa raíz

| Origen | Valor | Resuelve a |
| :--- | :--- | :--- |
| `model` (principal) | `opus` | `claude-opus-4-8` |
| `advisorModel` | `sonnet` | `claude-sonnet-4-6` |

Modelos incompatibles entre principal y advisor.

#### Solución

Hacer que ambos modelos sean coherentes. Cualquiera de estas opciones resuelve el 400:

1. Alinear el advisor al principal: `"advisorModel": "opus"`.
2. Bajar el principal: `"model": "sonnet"`.
3. Quitar `"advisorModel"` para que el advisor herede el modelo principal.

#### Lección

Cuando un error de modelo aparece en código que *no* define subagentes,
revisar la configuración heredada (`~/.claude/settings.json`, `.claude/settings.json`
del proyecto, `agents=`). El SDK aplica esa config aunque no se vea en la celda.


---

### MCP isError Flag
Cuando un MCP tool falla, debe comunicar el fallo usando el flag `isError`: true en la respuesta. Esto le indica al agente que la operación no fue exitosa y debe tomar una decisión de recovery.

```json
{
  "content": [
    {
      "type": "text",
      "text": "Customer not found: ID 'cust_999' does not exist in the active customer database."
    }
  ],
  "isError": true
}
```
Sin `isError`, el agente puede interpretar un mensaje de error como un resultado válido y continuar el flujo incorrectamente.



---

### Categorías de errores

No todos los errores son iguales. Clasificarlos permite al agente tomar decisiones de recovery apropiadas:

- **Transient (reintentar):** Fallos temporales que pueden resolverse con un retry.

- **Validation (corregir input):** El input proporcionado no cumple los requisitos.


- **Business (regla de negocio):** La operación viola una política o regla del dominio.

- **Permission (acceso denegado):** El caller no tiene permisos para la operación

---


### Cuando todos los errores retornan el mismo formato genérico:

```json
{
  "content": [{ "type": "text", "text": "Operation failed" }],
  "isError": true
}
```
El agente no puede decidir si debe reintentar, corregir el input, informar al usuario, o escalar. Esto causa loops de retry infinitos o abandonos prematuros.

### Structured Error Metadata
Incluye metadata estructurada que permita al agente tomar decisiones:

```json
{
  "content": [
    {
      "type": "text",
      "text": "Cannot apply 50% discount. Maximum allowed discount is 30% per company policy."
    }
  ],
  "isError": true,
  "_meta": {
    "errorCategory": "business",
    "isRetryable": false,
    "errorCode": "DISCOUNT_LIMIT_EXCEEDED",
    "maxAllowed": 30,
    "requested": 50
  }
}
```

| Campo | Tipo | Propósito / Descripción |
| :--- | :--- | :--- |
| **`errorCategory`** | `string` | Categoría del error: `transient`, `validation`, `business`, `permission`. |
| **`isRetryable`** | `boolean` | Indica si tiene sentido reintentar la operación automáticamente. |
| **`errorCode`** | `string` | Código máquina-legible diseñado para la lógica condicional del sistema. |
| **`description`** | `string` | Explicación amigable (human-readable) pensada para mostrar al usuario final. |

---

### Business rules: retriable false + explicación

Para errores de reglas de negocio, siempre marca isRetryable: false e incluye una explicación que el agente pueda transmitir al usuario:

```json
{
  "isError": true,
  "_meta": {
    "errorCategory": "business",
    "isRetryable": false,
    "description": "Las reservaciones deben hacerse con al menos 24 horas de anticipación."
  }
}
```
El agente sabe que no debe reintentar y tiene un mensaje claro para el usuario.

```json
{
  "content": [{ "type": "text", "text": "No results found for query 'xyz'" }],
  "isError": false,
  "_meta": { "resultCount": 0 }
}
```
Un `isError`: `false` con 0 resultados no es un error — es una respuesta válida que el agente debe aceptar sin reintentar.

In [19]:
import asyncio
from typing import Any
from claude_agent_sdk import (
    tool, create_sdk_mcp_server,
    query, ClaudeAgentOptions, ResultMessage,
)

# Estado para simular fallos transitorios
_intentos = {"n": 0}

@tool(
    name="get_inventory",
    description="Consulta el stock de un producto. Puede fallar transitoriamente.",
    input_schema={"product_id": str},
)
async def get_inventory(args: dict[str, Any]) -> dict[str, Any]:

    _intentos["n"] += 1

    print("Intento:", _intentos['n'])

    # Las primera vez: error TRANSIENT → el agente debería reintentar
    if _intentos["n"] < 2:
        response = {
            "content": [{"type": "text", "text": "Servicio temporalmente no disponible."}],
            "isError": True,
            "_meta": {"errorCategory": "transient", "isRetryable": True},
        }
        print(response)

        return response

    # Al 2do intento: éxito
    response = {
        "content": [{"type": "text", "text": f"Stock de {args['product_id']}: 42 unidades."}],
        "isError": False,
    }
    print(response)

    return response


inventory_server = create_sdk_mcp_server(
    name="inventory", version="1.0.0", tools=[get_inventory],
)

options = ClaudeAgentOptions(
    model = "claude-sonnet-4-6",
    system_prompt=(
        "Usás herramientas para consultar datos. Si una tool devuelve isError con "
        "errorCategory 'transient' (isRetryable true), reintentá la llamada. "
        "Si el error NO es retryable, no reintentes: informá al usuario."
    ),
    mcp_servers={"inventory": inventory_server},
    allowed_tools=["mcp__inventory__get_inventory"],
)

In [20]:
async for message in query(
    prompt="¿Cuánto stock hay del producto ABC-123?",
    options=options,
):
    if isinstance(message, ResultMessage) and message.subtype == "success":

        print("Respuesta final del modelo:\n", message.result)

Intento: 1
{'content': [{'type': 'text', 'text': 'Servicio temporalmente no disponible.'}], 'isError': True, '_meta': {'errorCategory': 'transient', 'isRetryable': True}}
Intento: 2
{'content': [{'type': 'text', 'text': 'Stock de ABC-123: 42 unidades.'}], 'isError': False}
Respuesta final del modelo:
 El producto **ABC-123** tiene actualmente **42 unidades** en stock.


---

## tool_choice: controlando la selección

El parámetro `tool_choice` en la API controla cómo Claude selecciona tools:

### 1. auto (default)
    Claude decide libremente si usar un tool o responder con texto:
```json
{ "tool_choice": { "type": "auto" } }
```
    Apropiado para: la mayoría de conversaciones donde el modelo debe decidir cuándo es relevante usar herramientas.


### 2. any (forzar alguna herramienta)
    Claude debe usar algún tool — no puede responder solo con texto:
```json
{ "tool_choice": { "type": "any" } }
```
    Apropiado para: pasos de un pipeline donde siempre se necesita una acción.


### 3. Forced (tool específico)
    Claude debe usar un tool particular:
```json
{ "tool_choice": { "type": "tool", "name": "search_database" } }
```
    Apropiado para: pasos predeterminados donde sabemos exactamente qué tool se necesita.

In [22]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

# "Base de datos" de mentira del usuario con el que se habla
USUARIO_ACTUAL = {"user_id": "u_001", "name": "Saúl Díaz", "email": "dsaul449@gmail.com"}


def get_user_data(user_id):
    # En producción: consultar tu DB por user_id.
    # Aquí devolvemos siempre el usuario actual para el ejemplo.
    return f"name={USUARIO_ACTUAL['name']}, email={USUARIO_ACTUAL['email']}"


tools = [
    {
        "name": "get_user_data",
        "description": "Devuelve el nombre y el email del usuario actual a partir de su user_id.",
        "input_schema": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "ID del usuario con el que se está hablando (ej: 'u_001').",
                }
            },
            "required": ["user_id"],
        },
    }
]


def process_tool_call(tool_name, tool_input):
    if tool_name == "get_user_data":
        return get_user_data(tool_input["user_id"])


def chat_with_claude(user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")

    messages = [{"role": "user", "content": user_message}]

    # PRIMERA llamada: FORZAMOS la tool get_user_data con tool_choice.
    # Sin importar lo que pida el usuario, Claude DEBE llamar esta tool.
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        messages=messages,
        tools=tools,
        tool_choice={"type": "tool", "name": "get_user_data"},  # ← FORCED
    )

    print("\nInitial Response:")
    print(f"Stop Reason: {message.stop_reason}")   # → 'tool_use', garantizado
    print(f"Content: {message.content}")

    while message.stop_reason != "end_turn":

        if message.stop_reason == "tool_use":
            tool_use = next(block for block in message.content if block.type == "tool_use")
            tool_name = tool_use.name
            tool_input = tool_use.input

            print(f"\nTool Used: {tool_name}")
            print(f"Tool Input: {tool_input}")

            tool_result = process_tool_call(tool_name, tool_input)

            print(f"Tool Result: {tool_result}")

            messages.extend(
                [
                    {
                        "role": "assistant",
                        "content": message.content,
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use.id,
                                "content": tool_result,
                            }
                        ],
                    },
                ]
            )

            # SIGUIENTES llamadas: tool_choice en AUTO (default).
            # Si dejáramos el forced aquí, Claude volvería a llamar la tool
            # en vez de redactar la respuesta final → loop infinito.
            message = client.messages.create(
                model=MODEL_NAME,
                max_tokens=4096,
                messages=messages,
                tools=tools,
            )

    final_response = next(
        (block.text for block in message.content if hasattr(block, "text")),
        None,
    )
    print(message.content)
    print(f"\nFinal Response: {final_response}")

    return final_response

In [23]:
chat_with_claude("Hola")


User Message: Hola

Initial Response:
Stop Reason: tool_use
Content: [ToolUseBlock(id='toolu_01PPrQQeiFw9BQfNuYQQkfFY', caller=DirectCaller(type='direct'), input={'user_id': '<UNKNOWN>'}, name='get_user_data', type='tool_use')]

Tool Used: get_user_data
Tool Input: {'user_id': '<UNKNOWN>'}
Tool Result: name=Saúl Díaz, email=dsaul449@gmail.com
[TextBlock(citations=None, text='¡Hola, **Saúl**! 👋 Bienvenido. ¿En qué puedo ayudarte hoy?', type='text')]

Final Response: ¡Hola, **Saúl**! 👋 Bienvenido. ¿En qué puedo ayudarte hoy?


'¡Hola, **Saúl**! 👋 Bienvenido. ¿En qué puedo ayudarte hoy?'